In [173]:
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('../src/')
import features.ptype_prepare_data as pp
import models.ptype_run_preds as rp
import models.score_classifier as score
import report.ptype_visualize as viz
import pandas as pd
import pickle
import os
import boto3
from datetime import datetime
from catboost import CatBoostClassifier
from sklearn.model_selection import cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve, f1_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay
import hydra
from omegaconf import DictConfig
from utils.logs import get_logger
import yaml
import data.s3_download_sso as sso_load
import data.clean_ceo_summary as ceo_clean

In [147]:
config_path='../params.yaml'
with open(config_path) as conf_file:
    config = yaml.safe_load(conf_file)
logger = get_logger('DATA_LOAD', log_level=config['base']['log_level'])

In [132]:
@hydra.main(config_path='params.yaml')
def main(config: DictConfig) -> None:
    print(config.data_load.sso_profile_name)

In [138]:
config['data_load']['data_prefix_list']

['slope', 's1', 's2', 'dem', 'features-ckpt-2023-02-09', 'labels']

In [174]:
sso_load.data_download(config_path)
ceo_summary = ceo_clean.import_ceo_summary(config_path)

2023-10-16 12:04:18,097 — DATA_DOWNLOAD — INFO — No new data downloaded
2023-10-16 12:04:18,131 — CEO_SUMMARY_LOAD — INFO — Summary data loaded


In [175]:
ceo_summary['plantations-train-v07']

{'Year Collected': nan,
 'Data Source': ['iiasa'],
 'Municipality': ['multiple'],
 'Country': ['brazil',
  'costa rica',
  'ivory coast',
  'ghana',
  'guatemala',
  'honduras'],
 'Purpose': ['prototyping'],
 'Vegetation categories (observed)': ['agroforestry'],
 'Plots in Polygons': nan,
 'Plots outside Polygons': 101.0,
 'Final plot count': nan,
 'Classes': ['binary'],
 'Non plantation (0)': nan,
 'Plantation (for mult-class this is mono) (1)': nan,
 'Agroforestry (2)': nan,
 'Plot ID': nan,
 'Use?': ['no'],
 'Notes': nan}

In [9]:
X, y = pp.create_xy(['v08'], 
                    classes='binary', 
                    drop_feats=False,
                    data_dir ='../data', 
                    verbose=False)


0.0 plots labeled unknown were dropped from v08.
warning needs to be updated
Plot id 08001 has no cloud free imagery and will be removed.
Plot id 08002 has no cloud free imagery and will be removed.
Plot id 08003 has no cloud free imagery and will be removed.
Plot id 08004 has no cloud free imagery and will be removed.
Plot id 08005 has no cloud free imagery and will be removed.
Plot id 08006 has no cloud free imagery and will be removed.
Plot id 08007 has no cloud free imagery and will be removed.
Plot id 08008 has no cloud free imagery and will be removed.
Plot id 08009 has no cloud free imagery and will be removed.
Plot id 08010 has no cloud free imagery and will be removed.
Plot id 08011 has no cloud free imagery and will be removed.
Plot id 08012 has no cloud free imagery and will be removed.
Plot id 08013 has no cloud free imagery and will be removed.
Plot id 08014 has no cloud free imagery and will be removed.
Plot id 08015 has no cloud free imagery and will be removed.
Plot id 

0it [00:00, ?it/s]

Class count {}
